In [1]:
from datasets import load_dataset
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

### Training BPE on wikitext


In [2]:
dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")

def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i: i+1000]["text"]

# BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# pre-tokenizer - can use options like WhiteSpace()
# Ġ - this represents a white-space character
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
print(f"Tokenized sentence: {tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!")}")


# Define trainer
trainer = trainers.BpeTrainer(vocab_size=25000, special_tokens=["<|endoftext|>", "[IMG]", "[UNK]"])
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

# Encode string
encoding = tokenizer.encode("Let's test the tokenizer")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# Define decoder
tokenizer.decoder = decoders.ByteLevel()
print(f"\nDecoded ids: {tokenizer.decode(encoding.ids)}")

Tokenized sentence: [('Let', (0, 3)), ("'s", (3, 5)), ('Ġtest', (5, 10)), ('Ġpre', (10, 14)), ('-', (14, 15)), ('tokenization', (15, 27)), ('!', (27, 28))]



Tokens: ['L', 'et', "'", 's', 'Ġtest', 'Ġthe', 'Ġto', 'ken', 'izer']
Output id: [46, 271, 9, 85, 2861, 200, 229, 3565, 11901]

Decoded ids: Let's test the tokenizer


### Training BPE on low, lower, lowest


In [3]:
# Decoding on custom dataset
tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

trainer = trainers.BpeTrainer(vocab_size=100)
tokenizer.train_from_iterator(['low', 'lower', 'lowest'], trainer=trainer)
print(f"Vocabulary learned: {tokenizer.get_vocab()}")

encoding = tokenizer.encode("lowering")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# The token IDs are dependent on the order characters are inserted into dictionary
# e gets 0 because e-w are inserted in alphabetical order.

# lowering is tokenized as: l o w e r i n g
# 1. lo
# 2. low
# 3. lowe
# 4. lower




Vocabulary learned: {'lowest': 12, 's': 4, 'low': 8, 'l': 1, 'lower': 11, 'e': 0, 'lowe': 9, 'lo': 7, 'r': 3, 'o': 2, 'w': 6, 'st': 10, 't': 5}
Tokens: ['lower']
Output id: [11]


In [5]:
# Decoding on custom dataset - with special token
tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

trainer = trainers.BpeTrainer(vocab_size=100, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(['low', 'lower', 'lowest'], trainer=trainer)
print(f"Vocabulary learned: {tokenizer.get_vocab()}")

encoding = tokenizer.encode("lower the lowest rope?")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# The token IDs are dependent on the order characters are inserted into dictionary
# e gets 0 because e-w are inserted in alphabetical order.

# lowering is tokenized as: l o w e r i n g
# 1. lo
# 2. low
# 3. lowe
# 4. lower




Vocabulary learned: {'[UNK]': 0, 't': 6, 'o': 3, 's': 5, 'r': 4, 'lowe': 10, 'lowest': 13, 'e': 1, 'l': 2, 'w': 7, 'st': 11, 'lower': 12, 'low': 9, 'lo': 8}
Tokens: ['lower', 't', 'e', 'lowest', 'r', 'o', 'e']
Output id: [12, 6, 1, 13, 4, 3, 1]


### Pre-processing

1. Normalization
2. Pre-tokenization


In [6]:
# Normalization
# Used to make raw strings cleaner - less random
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents, Strip, Lowercase

# combine normalizers with sequence
# normalizer_NFD = NFD()
# normalizer_SA = StripAccents()
# normalizer_S = Strip()
normalizer = normalizers.Sequence([NFD(), StripAccents(), Lowercase()])

print(f"Normalizer NFD + StripAccen + Lowercase: {normalizer.normalize_str("Héllo, how are yoü?")}")

Normalizer NFD + StripAccen + Lowercase: hello, how are you?


In [7]:
# Pre-tokenization
# Sets an upper bound to tokens after tokenizer is applied
# Ex: Split a string into words based on space - preventing a subword token from containing space
from tokenizers.pre_tokenizers import Whitespace, Digits

pre_tokenizer_WS = Whitespace()
pre_tokenizer_DIGITS = Digits()
pre_tokenizer_WS_DIGITS = pre_tokenizers.Sequence([Whitespace(), Digits()])

print(f"With white spaces: {pre_tokenizer_WS.pre_tokenize_str("Hello! how are you? Call 123")}")
print(f"\nWith digits: {pre_tokenizer_DIGITS.pre_tokenize_str("Hello! how are you? Call 123")}")
print(f"\nWith whitespace and digits: {pre_tokenizer_WS_DIGITS.pre_tokenize_str("Hello! how are you? Call 123")}")

With white spaces: [('Hello', (0, 5)), ('!', (5, 6)), ('how', (7, 10)), ('are', (11, 14)), ('you', (15, 18)), ('?', (18, 19)), ('Call', (20, 24)), ('123', (25, 28))]

With digits: [('Hello! how are you? Call ', (0, 25)), ('123', (25, 28))]

With whitespace and digits: [('Hello', (0, 5)), ('!', (5, 6)), ('how', (7, 10)), ('are', (11, 14)), ('you', (15, 18)), ('?', (18, 19)), ('Call', (20, 24)), ('123', (25, 28))]


### Model

1. models.BPE
2. models.Unigram
3. models.WordLevel
4. models.WordPiece


### Post-processing

Perform additional transformation to encoding - like adding a template, special tokens


### Questions

1. How does BPE work?
2. How does byte level BPE work?
3. What are those numbers in the ID?
4. What are normalizers? post processors?
5. How to use the template?
